# Raw XES dataframe view

Load the original BPI 2012 XES file into a pandas dataframe without the preprocessing used in `src/preprocess_bpi2012.py`.

Each row is one raw XES event. Trace-level attributes are prefixed with `trace:` and event-level attributes are prefixed with `event:`.

In [1]:
print('hello')

hello


In [3]:
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [5]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root containing data/ and src/.")


PROJECT_ROOT = find_project_root()
RAW_PATH = PROJECT_ROOT / "data" / "BPIC2012" / "BPI_Challenge_2012.xes"

print(PROJECT_ROOT)
print(RAW_PATH)
print(f"exists={RAW_PATH.exists()}, size_mb={RAW_PATH.stat().st_size / 1024 / 1024:.2f}")

C:\Users\hyewoo choi\Documents\99. ??숈썝\0. ?쇰Ц\git\time-aware-behavior-prediction
C:\Users\hyewoo choi\Documents\99. ??숈썝\0. ?쇰Ц\git\time-aware-behavior-prediction\data\BPIC2012\BPI_Challenge_2012.xes
exists=True, size_mb=70.67


In [6]:
def strip_namespace(tag: str) -> str:
    return tag.rsplit("}", 1)[-1] if "}" in tag else tag


def attributes_to_dict(parent: ET.Element, prefix: str) -> dict[str, object]:
    values: dict[str, object] = {}
    for child in parent:
        tag_type = strip_namespace(child.tag)
        key = child.attrib.get("key")
        if key is None:
            continue

        column = f"{prefix}:{key}"
        value = child.attrib.get("value")
        values[column] = value
        values[f"{column}__xes_type"] = tag_type

    return values


def load_raw_xes_dataframe(
    RAW_PATH: str | Path,
    max_traces: int | None = None,
    max_events: int | None = None,
) -> pd.DataFrame:
    RAW_PATH = Path(RAW_PATH)
    rows: list[dict[str, object]] = []
    trace_count = 0
    event_count = 0

    context = ET.iterparse(RAW_PATH, events=("end",))
    for _, elem in context:
        if strip_namespace(elem.tag) != "trace":
            continue

        trace_count += 1
        trace_attrs = attributes_to_dict(elem, "trace")

        for event_idx, event in enumerate(
            child for child in elem if strip_namespace(child.tag) == "event"
        ):
            row = {
                "raw_trace_idx": trace_count - 1,
                "raw_event_idx_in_trace": event_idx,
            }
            row.update(trace_attrs)
            row.update(attributes_to_dict(event, "event"))
            rows.append(row)

            event_count += 1
            if max_events is not None and event_count >= max_events:
                elem.clear()
                return pd.DataFrame(rows)

        elem.clear()
        if max_traces is not None and trace_count >= max_traces:
            break

    return pd.DataFrame(rows)

Set `MAX_TRACES` or `MAX_EVENTS` to a number if you only want a quick preview. Leave both as `None` to load the full original XES into a dataframe.

In [7]:
MAX_TRACES = 20
MAX_EVENTS = 50

raw_events = load_raw_xes_dataframe(
    RAW_PATH,
    max_traces=MAX_TRACES,
    max_events=MAX_EVENTS,
)

raw_events.shape

(50, 16)

In [9]:
type_columns = raw_events.columns[raw_events.columns.str.endswith("_type")]
raw_events_without_type = raw_events.drop(columns=type_columns)

print(f"excluded_type_columns={len(type_columns)}")
raw_events_without_type.shape

excluded_type_columns=7


(50, 9)

In [10]:
raw_events_without_type.head(20)

,raw_trace_idx,raw_event_idx_in_trace,trace:REG_DATE,trace:concept:name,trace:AMOUNT_REQ,event:org:resource,event:lifecycle:transition,event:concept:name,event:time:timestamp
0,0,0,2011-10-01T00:38:44.546+02:00,173688,20000,112,COMPLETE,A_SUBMITTED,2011-10-01T00:38:44.546+02:00
1,0,1,2011-10-01T00:38:44.546+02:00,173688,20000,112,COMPLETE,A_PARTLYSUBMITTED,2011-10-01T00:38:44.880+02:00
2,0,2,2011-10-01T00:38:44.546+02:00,173688,20000,112,COMPLETE,A_PREACCEPTED,2011-10-01T00:39:37.906+02:00
3,0,3,2011-10-01T00:38:44.546+02:00,173688,20000,112,SCHEDULE,W_Completeren aanvraag,2011-10-01T00:39:38.875+02:00
4,0,4,2011-10-01T00:38:44.546+02:00,173688,20000,NaN,START,W_Completeren aanvraag,2011-10-01T11:36:46.437+02:00
5,0,5,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,A_ACCEPTED,2011-10-01T11:42:43.308+02:00
6,0,6,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,O_SELECTED,2011-10-01T11:45:09.243+02:00
7,0,7,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,A_FINALIZED,2011-10-01T11:45:09.243+02:00
8,0,8,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,O_CREATED,2011-10-01T11:45:11.197+02:00
9,0,9,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,O_SENT,2011-10-01T11:45:11.380+02:00


In [11]:
raw_events_without_type.dtypes

raw_trace_idx                 int64
raw_event_idx_in_trace        int64
trace:REG_DATE                  str
trace:concept:name              str
trace:AMOUNT_REQ                str
event:org:resource              str
event:lifecycle:transition      str
event:concept:name              str
event:time:timestamp            str
dtype: object

In [12]:
raw_events_without_type.columns.to_list()

['raw_trace_idx',
 'raw_event_idx_in_trace',
 'trace:REG_DATE',
 'trace:concept:name',
 'trace:AMOUNT_REQ',
 'event:org:resource',
 'event:lifecycle:transition',
 'event:concept:name',
 'event:time:timestamp']

In [13]:
raw_events_without_type.filter(regex=r"^(trace|event):").head(20)

,trace:REG_DATE,trace:concept:name,trace:AMOUNT_REQ,event:org:resource,event:lifecycle:transition,event:concept:name,event:time:timestamp
0,2011-10-01T00:38:44.546+02:00,173688,20000,112,COMPLETE,A_SUBMITTED,2011-10-01T00:38:44.546+02:00
1,2011-10-01T00:38:44.546+02:00,173688,20000,112,COMPLETE,A_PARTLYSUBMITTED,2011-10-01T00:38:44.880+02:00
2,2011-10-01T00:38:44.546+02:00,173688,20000,112,COMPLETE,A_PREACCEPTED,2011-10-01T00:39:37.906+02:00
3,2011-10-01T00:38:44.546+02:00,173688,20000,112,SCHEDULE,W_Completeren aanvraag,2011-10-01T00:39:38.875+02:00
4,2011-10-01T00:38:44.546+02:00,173688,20000,NaN,START,W_Completeren aanvraag,2011-10-01T11:36:46.437+02:00
5,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,A_ACCEPTED,2011-10-01T11:42:43.308+02:00
6,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,O_SELECTED,2011-10-01T11:45:09.243+02:00
7,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,A_FINALIZED,2011-10-01T11:45:09.243+02:00
8,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,O_CREATED,2011-10-01T11:45:11.197+02:00
9,2011-10-01T00:38:44.546+02:00,173688,20000,10862,COMPLETE,O_SENT,2011-10-01T11:45:11.380+02:00
